In [3]:
!pip install pandas
# pulp

  Using cached pandas-3.0.2-cp312-cp312-macosx_10_13_x86_64.whl.metadata (79 kB)
Using cached pandas-3.0.2-cp312-cp312-macosx_10_13_x86_64.whl (10.3 MB)


In [4]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

In [6]:
# prices = pd.read_csv('/Users/marysia/Desktop/applied/OR/food_prices_200.xlsx - food_prices.csv')
prices = pd.read_csv('/Users/julianyoo/Downloads/Study/PWR/Term 1/Operations Research/food_prices_200.xlsx - food_prices.csv')
# nutrients = pd.read_csv('/Users/marysia/Desktop/applied/OR/food_nutrition_200.xlsx - food_nutrition_per_100g.csv')
nutrients = pd.read_csv('/Users/julianyoo/Downloads/Study/PWR/Term 1/Operations Research/food_nutrition_200.xlsx - food_nutrition_per_100g.csv')

In [6]:
prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   food_id                 205 non-null    int64  
 1   food_name               205 non-null    object 
 2   category                205 non-null    object 
 3   subcategory             205 non-null    object 
 4   market_unit             205 non-null    object 
 5   price_per_unit_usd      205 non-null    float64
 6   unit_weight_g           205 non-null    object 
 7   price_per_100g_usd      205 non-null    float64
 8   price_per_1g_usd        205 non-null    float64
 9   bls_series_or_source    205 non-null    object 
 10  price_reference_period  204 non-null    object 
 11  official_data_url       205 non-null    object 
 12  notes                   205 non-null    object 
dtypes: float64(3), int64(1), object(9)
memory usage: 20.9+ KB


In [5]:
nutrients.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 35 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   food_id                205 non-null    int64  
 1   food_name              205 non-null    object 
 2   category               205 non-null    object 
 3   subcategory            205 non-null    object 
 4   fdc_id                 205 non-null    int64  
 5   preparation_state      205 non-null    object 
 6   energy_kcal            205 non-null    int64  
 7   protein_g              205 non-null    float64
 8   total_fat_g            205 non-null    float64
 9   carbohydrates_g        205 non-null    float64
 10  dietary_fiber_g        205 non-null    float64
 11  sugars_g               205 non-null    float64
 12  saturated_fat_g        205 non-null    float64
 13  monounsaturated_fat_g  205 non-null    float64
 14  polyunsaturated_fat_g  205 non-null    float64
 15  choles

In [7]:
# required = pd.read_csv('/Users/marysia/Desktop/applied/OR/last_version_prpcessed.csv')
required = pd.read_csv('/Users/julianyoo/Downloads/Study/PWR/Term 1/Operations Research/OR_Project - final_or_ill_cry.csv')

In [8]:
required.head()

,Nutrient,7–9 years,10–12 years,13–15 years,lower_7-9,lower_10-12,lower_13-15,upper_7-9,upper_10-12,upper_13-15
0,Energy [kcal/day],1800.0,2200.0,2700.0,1500.0,1900.0,2500.0,1800.0,2200.0,2700.0
1,Protein [g/day],30.0,42.0,58.0,25.0,35.0,50.0,35.0,50.0,75.0
2,Sodium [mg/day],1200.0,1300.0,1500.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Carbohydrates [% energy],50.0,50.0,50.0,45.0,45.0,45.0,65.0,65.0,65.0
4,Fiber [g/day],16.0,19.0,19.0,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# Old version of the code, ignore
import pandas as pd
from pulp import *

# 1. WCZYTANIE DANYCH
df_norms = pd.read_csv('last_version_prpcessed.csv')
df_nutrition = pd.read_csv('food_nutrition_200.xlsx - food_nutrition_per_100g.csv')
df_prices = pd.read_csv('food_prices_200.xlsx - food_prices.csv')

# 2. OCZYSZCZANIE DANYCH (Kluczowe dla uniknięcia TypeError)
# Lista kolumn z wartościami odżywczymi do konwersji
nutrient_cols = ['energy_kcal', 'protein_g', 'total_fat_g', 'carbohydrates_g', 'sodium_mg', 'dietary_fiber_g']

# Konwersja kolumn w produktach na liczby, błędy zamieniamy na 0
for col in nutrient_cols:
    df_nutrition[col] = pd.to_numeric(df_nutrition[col], errors='coerce').fillna(0)

# Konwersja ceny na liczby
df_prices['price_per_100g_usd'] = pd.to_numeric(df_prices['price_per_100g_usd'], errors='coerce').fillna(0)

# Połączenie cen z wartościami odżywczymi
df_products = pd.merge(df_nutrition, df_prices[['food_id', 'price_per_100g_usd']], on='food_id')

def solve_diet_simplex(age_group_label):
    # Oczyszczenie nazwy grupy i normalizacja myślnika
    clean_label = age_group_label.strip()
    suffix = clean_label.split(' ')[0].replace('–', '-') 
    
    # Konwersja kolumn w normach na liczby (dla wybranej grupy)
    cols_to_fix = [age_group_label, f'lower_{suffix}', f'upper_{suffix}']
    for c in cols_to_fix:
        if c in df_norms.columns:
            df_norms[c] = pd.to_numeric(df_norms[c], errors='coerce')

    # Pobranie zapotrzebowania kalorycznego
    energy_row = df_norms[df_norms['Nutrient'].str.contains('Energy', na=False)]
    total_kcal = float(energy_row[age_group_label].values[0])

    # Inicjalizacja modelu
    prob = LpProblem(f"Dieta_{suffix}", LpMinimize)

    # Zmienne: ilość gramów produktu (limit 500g na produkt)
    food_items = df_products['food_name'].tolist()
    food_vars = LpVariable.dicts("gram", food_items, lowBound=0, upBound=500, cat='Continuous')

    # Funkcja celu: koszt
    prob += lpSum([(food_vars[row['food_name']] / 100) * row['price_per_100g_usd'] for _, row in df_products.iterrows()])

    # Mapowanie składników
    nutrient_map = {
        'Energy [kcal/day]': ('energy_kcal', 1),
        'Protein [g/day]': ('protein_g', 1),
        'Fat [% energy]': ('total_fat_g', 9),
        'Carbohydrates [% energy]': ('carbohydrates_g', 4),
        'Sodium [mg/day]': ('sodium_mg', 1),
        'Fiber [g/day]': ('dietary_fiber_g', 1)
    }

    print(f"\n--- Group: {clean_label} ({total_kcal} kcal) ---")

    for norm_name, (food_col, kcal_factor) in nutrient_map.items():
        if norm_name not in df_norms['Nutrient'].values:
            continue
            
        row = df_norms[df_norms['Nutrient'] == norm_name].iloc[0]
        prop_val = float(row[age_group_label])
        
        # Pobieranie limitów
        lb_val = row.get(f'lower_{suffix}')
        ub_val = row.get(f'upper_{suffix}')
        
        # Twoja logika limitów (0.3 / 1.0)
        final_lb = float(lb_val) if pd.notnull(lb_val) else prop_val * 0.3
        final_ub = float(ub_val) if pd.notnull(ub_val) else prop_val * 1.0

        # Przeliczenie % na gramy
        if '% energy' in norm_name:
            final_lb = (final_lb * total_kcal / 100) / kcal_factor
            final_ub = (final_ub * total_kcal / 100) / kcal_factor

        # Dodanie ograniczenia: suma składnika w diecie musi być w przedziale [LB, UB]
        total_in_diet = lpSum([(food_vars[r['food_name']] / 100) * float(r[food_col]) for _, r in df_products.iterrows()])
        
        prob += total_in_diet >= final_lb, f"Min_{food_col}"
        prob += total_in_diet <= final_ub, f"Max_{food_col}"

    # Rozwiązanie
    status = prob.solve(PULP_CBC_CMD(msg=0))

    if LpStatus[status] == 'Optimal':
        results = []
        for item in food_items:
            val = food_vars[item].varValue
            if val and val > 0.1:
                results.append({"Product": item, "Grams": round(val, 2)})
        
        print(pd.DataFrame(results).to_string(index=False))
        print(f"Cost: ${round(value(prob.objective), 2)}")
    else:
        print("Error: diet within contrains not found")

# Uruchomienie
groups = ['7–9 years ', '10–12 years ', '13–15 years']
for g in groups:
    solve_diet_simplex(g)


--- Group: 7–9 years (1800.0 kcal) ---
                            Product  Grams
    Pasta, Spaghetti, Dry, Enriched  40.90
                    Watermelon, Raw 500.00
                Mayonnaise, Regular  42.79
                         Water, Tap 500.00
Cornmeal, Whole Grain, Yellow (dry) 173.85
Cost: $0.93

--- Group: 10–12 years (2200.0 kcal) ---
                             Product  Grams
     Pasta, Spaghetti, Dry, Enriched  73.54
                     Watermelon, Raw 500.00
Vegetable Oil (Soybean/Canola blend)   5.27
                 Mayonnaise, Regular  45.73
                          Water, Tap 500.00
 Cornmeal, Whole Grain, Yellow (dry) 200.64
Cost: $1.15

--- Group: 13–15 years (2700.0 kcal) ---
                             Product  Grams
     Pasta, Spaghetti, Dry, Enriched 210.60
                     Watermelon, Raw 500.00
Vegetable Oil (Soybean/Canola blend)   7.89
                 Mayonnaise, Regular  57.19
                          Water, Tap 500.00
 Cornmeal, Whole Grain

In [ ]:
# new code, don't ignore
import pandas as pd
from pulp import *

# 1. LOAD DATA
df_norms = required
df_nutrition = nutrients 
df_prices = prices

# 2. DATA CLEANING
nutrient_cols = ['energy_kcal', 'protein_g', 'total_fat_g', 'carbohydrates_g', 'sodium_mg', 'dietary_fiber_g']
for col in nutrient_cols:
    df_nutrition[col] = pd.to_numeric(df_nutrition[col], errors='coerce').fillna(0)

df_prices['price_per_100g_usd'] = pd.to_numeric(df_prices['price_per_100g_usd'], errors='coerce').fillna(0)

# Merge nutrition with prices and include category for the Fruit/Veg constraint
df_products = pd.merge(df_nutrition, df_prices[['food_id', 'price_per_100g_usd']], on='food_id')

def solve_lunch_simplex(age_group_label):
    clean_label = age_group_label.strip()
    suffix = clean_label.split(' ')[0].replace('–', '-') 
    
    # Scaling factor for Lunch (35% of daily intake)
    LUNCH_SCALE = 0.35
    
    # Fix norms columns
    cols_to_fix = [age_group_label, f'lower_{suffix}', f'upper_{suffix}']
    for c in cols_to_fix:
        if c in df_norms.columns:
            df_norms[c] = pd.to_numeric(df_norms[c], errors='coerce')

    energy_row = df_norms[df_norms['Nutrient'].str.contains('Energy', na=False)]
    total_kcal_day = float(energy_row[age_group_label].values[0])
    lunch_kcal = total_kcal_day * LUNCH_SCALE

    # Initialize model
    prob = LpProblem(f"Lunch_{suffix}", LpMinimize)

    # Variables: 
    # x = grams of food (Max 100g)
    # y = binary variable (1 if food is chosen, 0 otherwise)
    food_items = df_products['food_name'].tolist()
    food_vars = LpVariable.dicts("gram", food_items, lowBound=0, upBound=100, cat='Continuous')
    chosen_vars = LpVariable.dicts("chosen", food_items, cat='Binary')

    # Objective: Minimize cost
    prob += lpSum([(food_vars[item] / 100) * df_products.loc[df_products['food_name'] == item, 'price_per_100g_usd'].values[0] for item in food_items])

    # Constraint Logic: Link x and y (If x > 0, y must be 1)
    for item in food_items:
        prob += food_vars[item] <= 100 * chosen_vars[item]
        prob += food_vars[item] >= 0.1 * chosen_vars[item] # Min 0.1g if chosen to avoid tiny fractions

    # 3. DIVERSITY: At least 5 different ingredients
    prob += lpSum([chosen_vars[item] for item in food_items]) >= 5

    # 4. FRUIT/VEG: At least 1 product from Fruit or Vegetable category
    veg_fruit_items = df_products[df_products['category'].str.contains('Fruit|Vegetable', case=False, na=False)]['food_name'].tolist()
    prob += lpSum([chosen_vars[item] for item in veg_fruit_items]) >= 1

    # 5. NUTRITIONAL CONSTRAINTS (Scaled by 35%)
    nutrient_map = {
        'Energy [kcal/day]': ('energy_kcal', 1),
        'Protein [g/day]': ('protein_g', 1),
        'Fat [% energy]': ('total_fat_g', 9),
        'Carbohydrates [% energy]': ('carbohydrates_g', 4),
        'Sodium [mg/day]': ('sodium_mg', 1),
        'Fiber [g/day]': ('dietary_fiber_g', 1)
    }

    print(f"\n--- LUNCH: {clean_label} ({round(lunch_kcal, 1)} kcal target) ---")

    for norm_name, (food_col, kcal_factor) in nutrient_map.items():
        if norm_name not in df_norms['Nutrient'].values:
            continue
            
        row = df_norms[df_norms['Nutrient'] == norm_name].iloc[0]
        prop_val = float(row[age_group_label])
        
        lb_val = row.get(f'lower_{suffix}')
        ub_val = row.get(f'upper_{suffix}')
        
        # Apply 30-100% logic to daily norm, then scale for lunch
        final_lb = (float(lb_val) if pd.notnull(lb_val) else prop_val * 0.3) * LUNCH_SCALE
        final_ub = (float(ub_val) if pd.notnull(ub_val) else prop_val * 1.0) * LUNCH_SCALE

        if '% energy' in norm_name:
            # For percentages, we calculate the gram requirement based on the LUNCH calories
            final_lb = (float(lb_val) if pd.notnull(lb_val) else prop_val * 0.3)
            final_ub = (float(ub_val) if pd.notnull(ub_val) else prop_val * 1.0)
            final_lb = (final_lb * lunch_kcal / 100) / kcal_factor
            final_ub = (final_ub * lunch_kcal / 100) / kcal_factor

        total_in_diet = lpSum([(food_vars[r['food_name']] / 100) * float(r[food_col]) for _, r in df_products.iterrows()])
        
        prob += total_in_diet >= final_lb, f"Min_{food_col}"
        prob += total_in_diet <= final_ub, f"Max_{food_col}"

    # Solve
    status = prob.solve(PULP_CBC_CMD(msg=0))

    if LpStatus[status] == 'Optimal':
        results = []
        for item in food_items:
            val = food_vars[item].varValue
            if val and val > 0.1:
                cat = df_products.loc[df_products['food_name'] == item, 'category'].values[0]
                results.append({"Product": item, "Category": cat, "Grams": round(val, 2)})
        
        print(pd.DataFrame(results).to_string(index=False))
        print(f"Lunch Cost: ${round(value(prob.objective), 2)}")
    else:
        print("ERROR: Could not find a lunch meeting these constraints. Try loosening the 100g limit.")

# Execution
groups = ['7–9 years', '10–12 years', '13–15 years']
for g in groups:
    solve_lunch_simplex(g)


--- LUNCH: 7–9 years (630.0 kcal target) ---
                             Produkt   Kategoria  Gramy
               Whole Chicken (fresh)     Poultry   1.07
     Pasta, Spaghetti, Dry, Enriched      Grains  20.51
                     Watermelon, Raw      Fruits 100.00
Vegetable Oil (Soybean/Canola blend) Fats & Oils   9.88
                 Mayonnaise, Regular  Condiments  15.29
                          Water, Tap   Beverages 100.00
 Cornmeal, Whole Grain, Yellow (dry)      Grains  62.24
Lunch Cost: $0.38

--- LUNCH: 10–12 years (770.0 kcal target) ---
                             Produkt   Kategoria  Gramy
               Whole Chicken (fresh)     Poultry   9.49
     Pasta, Spaghetti, Dry, Enriched      Grains  31.95
                     Watermelon, Raw      Fruits 100.00
Vegetable Oil (Soybean/Canola blend) Fats & Oils  15.07
                 Mayonnaise, Regular  Condiments  15.23
                          Water, Tap   Beverages 100.00
 Cornmeal, Whole Grain, Yellow (dry)      Grains

In [18]:
import pandas as pd
from pulp import *

# 1. LOAD DATA
df_norms = pd.read_csv('OR_Project - final_or_ill_cry.csv')
df_nutrition = pd.read_csv('food_nutrition_200.xlsx - food_nutrition_per_100g.csv')
df_prices = pd.read_csv('food_prices_200.xlsx - food_prices.csv')

# Cleaning and Merging
nutrient_cols = df_nutrition.columns[6:]
for col in nutrient_cols:
    df_nutrition[col] = pd.to_numeric(df_nutrition[col], errors='coerce').fillna(0)
df_prices['price_per_100g_usd'] = pd.to_numeric(df_prices['price_per_100g_usd'], errors='coerce').fillna(0)
df_products = pd.merge(df_nutrition, df_prices[['food_id', 'price_per_100g_usd']], on='food_id')

# Item Groups
beverages = df_products[df_products['category'] == 'Beverages']['food_name'].tolist()
dairy_items = df_products[df_products['category'] == 'Dairy']['food_name'].tolist()

# Nutrient Mapping
nutrient_map = {
    'Energy [kcal/day]': ('energy_kcal', 1),
    'Protein [g/day]': ('protein_g', 1),
    'Fat [% energy]': ('total_fat_g', 9),
    'Carbohydrates [% energy]': ('carbohydrates_g', 4),
    'Fiber [g/day]': ('dietary_fiber_g', 1),
    'Sodium [mg/day]': ('sodium_mg', 1),
    'Vitamin A [μg/day]': ('vitamin_a_ug_rae', 1),
    'Vitamin D [μg/day]': ('vitamin_d_ug', 1),
    'Vitamin E [mg/day]': ('vitamin_e_mg_ate', 1),
    'Vitamin K [μg/day]': ('vitamin_k_ug', 1),
    'Vitamin C [mg/day]': ('vitamin_c_mg', 1),
    'Thiamine [mg/day]': ('thiamin_b1_mg', 1),
    'Riboflavin [mg/day]': ('riboflavin_b2_mg', 1),
    'Niacin [mg/day]': ('niacin_b3_mg', 1),
    'Vitamin B6 [mg/day]': ('vitamin_b6_mg', 1),
    'Folates [μg/day]': ('folate_ug_dfe', 1),
    'Cobalamin [μg/day]': ('vitamin_b12_ug', 1),
    'Calcium [mg/day]': ('calcium_mg', 1),
    'Phosphorus [mg/day]': ('phosphorus_mg', 1),
    'Magnesium [mg/day]': ('magnesium_mg', 1),
    'Iron [mg/day]': ('iron_mg', 1),
    'Zinc [mg/day]': ('zinc_mg', 1),
    'Selenium [μg/day]': ('selenium_ug', 1),
    'Potassium [mg/day]': ('potassium_mg', 1)
}

def solve_lunch_optimization(age_group_label, lb_f=0.3, ub_f=0.35):
    suffix = age_group_label.strip().split(' ')[0].replace('–', '-')
    prob = LpProblem(f"Lunch_Optimization_{suffix}", LpMinimize)
    
    food_names = df_products['food_name'].tolist()
    food_vars = LpVariable.dicts("gram", food_names, lowBound=0, upBound=400, cat='Continuous')
    chosen_vars = LpVariable.dicts("chosen", food_names, cat='Binary')

    # Objective
    prob += lpSum([(food_vars[row['food_name']] / 100) * row['price_per_100g_usd'] for _, row in df_products.iterrows()])

    # Logic Constraints
    for _, row in df_products.iterrows():
        name = row['food_name']
        is_beverage = row['category'] == 'Beverages'
        
        # Upper Bound logic: 100g for food, 400g for drinks
        limit_high = 400 if is_beverage else 100
        
        prob += food_vars[name] >= 10 * chosen_vars[name] # Min 10g
        prob += food_vars[name] <= limit_high * chosen_vars[name] # Max weight

    # Limit categories to 1 item each
    prob += lpSum([chosen_vars[b] for b in beverages]) <= 1, "Max_One_Beverage"
    prob += lpSum([chosen_vars[d] for d in dairy_items]) <= 1, "Max_One_Dairy"

    # Nutritional Constraints
    daily_energy = float(df_norms[df_norms['Nutrient'] == 'Energy [kcal/day]'].iloc[0][age_group_label])

    for norm_name, (food_col, kcal_factor) in nutrient_map.items():
        if norm_name not in df_norms['Nutrient'].values: continue
        row = df_norms[df_norms['Nutrient'] == norm_name].iloc[0]
        
        # Get base values
        prop = float(row[age_group_label])
        low = float(row[f'lower_{suffix}']) if pd.notnull(row[f'lower_{suffix}']) else prop
        high = float(row[f'upper_{suffix}']) if pd.notnull(row[f'upper_{suffix}']) else prop

        # Calculate meal-specific bounds
        final_lb = low * lb_f
        final_ub = high * ub_f

        if '% energy' in norm_name:
            final_lb = (final_lb * daily_energy / 100) / kcal_factor
            final_ub = (final_ub * daily_energy / 100) / kcal_factor

        actual_sum = lpSum([(food_vars[r['food_name']] / 100) * r[food_col] for _, r in df_products.iterrows()])
        prob += actual_sum >= final_lb, f"Min_{food_col}"
        prob += actual_sum <= final_ub, f"Max_{food_col}"

    # Solve and recursive loosening
    status = prob.solve(PULP_CBC_CMD(msg=0))
    if LpStatus[status] != 'Optimal':
        if lb_f <= 0.15: # Stop at 15% of daily requirement
            print(f"FAILED for {age_group_label}. Constraints too tight.")
            return None
        
        # Loosen: Decrease min requirement, Increase max allowance
        print(f"Infeasible for {age_group_label}. Trying LB={round(lb_f-0.02, 2)}, UB={round(ub_f+0.05, 2)}...")
        return solve_lunch_optimization(age_group_label, lb_f - 0.02, ub_f + 0.05)

    # Output
    print(f"\n--- SUCCESS: Optimized Lunch for {age_group_label} ---")
    print(f"Final Factors: LB={round(lb_f, 2)}, UB={round(ub_f, 2)} | Cost: ${round(value(prob.objective), 2)}")
    results = [{"Product": item, "Grams": round(food_vars[item].varValue, 1)} 
               for item in food_names if food_vars[item].varValue > 0.1]
    print(pd.DataFrame(results).to_string(index=False))

# Run for groups
for group in ['7–9 years', '10–12 years', '13–15 years']:
    solve_lunch_optimization(group)


--- SUCCESS: Optimized Lunch for 7–9 years ---
Final Factors: LB=0.3, UB=0.35 | Cost: $1.47
                               Product  Grams
        Catfish, Farmed (fresh fillet)   18.3
            Canned Chickpeas (drained)   20.3
                      Tofu, Firm (raw)   19.5
Sunflower Seeds (kernels, dry-roasted)   10.0
 Milk, Almond, Unsweetened (fortified)   92.8
      Rice, White, Long Grain (cooked)   17.6
         Corn Flakes Cereal (enriched)   10.0
                   Oranges, Navel, Raw   17.2
                     Raisins, Seedless   21.6
         Apple Juice, from Concentrate   71.2
     Margarine, Regular (tub, 80% fat)   11.1
     Sports Drink (Gatorade, original)   51.7
           Corn Tortilla Chips (plain)   10.0
             Grapefruit, Pink/Red, Raw   10.0
Infeasible for 10–12 years. Trying LB=0.28, UB=0.4...

--- SUCCESS: Optimized Lunch for 10–12 years ---
Final Factors: LB=0.28, UB=0.4 | Cost: $1.18
                                  Product  Grams
           Catfish,